# Download and Load the Data

In [22]:
import torch
from torchvision import datasets
from torchvision import transforms

transform = transforms.ToTensor()

train_and_valid_data = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Inspect the Data

In [23]:
print(len(train_and_valid_data))
print(len(test_data))

60000
10000


# Split train data to train and validation data

In [24]:
torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [54000, 6000])

# Create the loaders

In [25]:
from torch.utils.data import DataLoader

torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

# Create the CNN

In [26]:
import torch.nn as nn
from functools import partial

torch.manual_seed(42)  # extra code – ensure reproducibility
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")
model = nn.Sequential(
    DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=64, out_channels=128), nn.ReLU(),
    DefaultConv2d(in_channels=128, out_channels=128), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(),
    DefaultConv2d(in_channels=256, out_channels=256), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(in_features=2304, out_features=128), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=128, out_features=64), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=64, out_features=10),
).to("cuda")

# Training with Early Stopping

In [27]:
import torchmetrics
import copy

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to("cuda"), y_batch.to("cuda")
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs):
    best_state_dict = copy.deepcopy(model.state_dict())
    best_valid_accuracy = 0
    patience = 10
    epochs_without_improvement = 0

    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to("cuda"), y_batch.to("cuda")
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        valid_metric_value = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(valid_metric_value)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")

        if(valid_metric_value > best_valid_accuracy):
            best_valid_accuracy = valid_metric_value
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print("Early stopping triggered.")
                break

    model.load_state_dict(best_state_dict)
    return history

n_epochs = 40
optimizer = torch.optim.AdamW(model.parameters())
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to("cuda")
history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)

Epoch 1/40, train loss: 0.6264, train metric: 0.7808, valid metric: 0.9712
Epoch 2/40, train loss: 0.1878, train metric: 0.9476, valid metric: 0.9807
Epoch 3/40, train loss: 0.1246, train metric: 0.9685, valid metric: 0.9862
Epoch 4/40, train loss: 0.0960, train metric: 0.9751, valid metric: 0.9895
Epoch 5/40, train loss: 0.0876, train metric: 0.9778, valid metric: 0.9843
Epoch 6/40, train loss: 0.0702, train metric: 0.9819, valid metric: 0.9875
Epoch 7/40, train loss: 0.0678, train metric: 0.9835, valid metric: 0.9855
Epoch 8/40, train loss: 0.0596, train metric: 0.9849, valid metric: 0.9907
Epoch 9/40, train loss: 0.0507, train metric: 0.9871, valid metric: 0.9895
Epoch 10/40, train loss: 0.0499, train metric: 0.9874, valid metric: 0.9922
Epoch 11/40, train loss: 0.0443, train metric: 0.9889, valid metric: 0.9832
Epoch 12/40, train loss: 0.0432, train metric: 0.9894, valid metric: 0.9913
Epoch 13/40, train loss: 0.0362, train metric: 0.9907, valid metric: 0.9888
Epoch 14/40, train lo

# Testing

In [28]:
evaluate_tm(model, test_loader, accuracy)

tensor(0.9933, device='cuda:0')